# Module 03 — LLM Inference

Run the agent loop with a real (or mocked) LLM.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../../src'))
from openenv_env import CodeDebugEnvironment, Action
from openenv_env.graders import GRADER_REGISTRY
from openenv_env.tasks import TASK_REGISTRY

## Environment Variable Check

In [ ]:
import os
for var in ["API_BASE_URL", "MODEL_NAME", "HF_TOKEN"]:
    val = os.getenv(var, "NOT SET")
    print(f"{var:15s}: {val[:40] if val != "NOT SET" else val}")

## Mock Agent (no API key needed)

In [ ]:
class MockAgent:
    """Always returns the reference fix — useful for testing the harness."""
    def act(self, buggy_code, error_message, hints):
        # In real use: call LLM here
        return Action(
            fixed_code=buggy_code + "  # mock fix",
            explanation="Mock agent returning code unchanged.",
            confidence=0.3
        )

agent = MockAgent()
print("Mock agent ready.")

## Run a Single Task Episode

In [ ]:
env = CodeDebugEnvironment(random_seed=1)
task_id = "task_syntax_001"
obs = env.reset(task_id=task_id)

for step in range(3):
    action = agent.act(obs.buggy_code, obs.error_message, obs.hints)
    scores = {name: fn(action, TASK_REGISTRY[task_id]) for name, fn in GRADER_REGISTRY.items()}
    result = env.step(action)
    print(f"Step {step+1}: composite={scores['composite']:.3f}  done={result.done}")
    obs = result.observation
    if result.done:
        break

## Structured Log Format

In [ ]:
import time

def log_start(task_id, model):
    """Emit [START] line — tag must be the first characters on the line."""
    print(f"[START] task={task_id} model={model}", flush=True)

def log_step(task_id, step, reward, done):
    """Emit [STEP] line."""
    print(f"[STEP] task={task_id} step={step} reward={reward:.4f} done={done}", flush=True)

def log_end(task_id, score, steps, solved):
    """Emit [END] line."""
    print(f"[END] task={task_id} score={score:.4f} steps={steps} solved={solved}", flush=True)

# Demo
log_start("task_syntax_001", "gpt-4o-mini")
log_step("task_syntax_001", 1, 0.6971, False)
log_step("task_syntax_001", 2, 0.9553, True)
log_end("task_syntax_001", 0.9553, 2, True)

## Parse Structured Logs

In [ ]:
import subprocess, sys, os

result = subprocess.run(
    [sys.executable, "../../inference.py",
     "--tasks", "task_syntax_001", "--max-steps", "1"],
    capture_output=True, text=True,
    cwd=os.path.dirname(os.path.abspath("../../inference.py"))
)

print("--- stdout ---")
for line in result.stdout.splitlines():
    if line.startswith(("[START]", "[STEP]", "[END]", "[SUMMARY]")):
        tag = line.split("]")[0] + "]"
        rest = line[len(tag):].strip()
        print(f"  tag={tag:10s}  data={rest}")

## Exercise
Set your `API_BASE_URL`, `MODEL_NAME`, and `HF_TOKEN` env vars,
then replace `MockAgent` with the real `LLMAgent` from `inference.py`.
Compare the composite scores between mock and LLM agent.